In [35]:
import requests
import json
import pandas as pd


In [50]:
BASE = "https://gamma-api.polymarket.com/events"

def fetch_events_page(offset, limit = 100):
    response = requests.get(
        BASE,
        params={
            "closed": "true",
            "offset": offset,
            "limit": limit,
            "order": "closedTime",
            "ascending": "false"
        }
    )
    return response.json()

events = fetch_events_page(0, 5);

In [51]:
def is_resolved(market):
    x = market["outcomePrices"]
    parsed = json.loads(x)
    numbers = [float(p) for p in parsed]
    if (numbers[0] > 0.99 and numbers[1] < 0.01) or (numbers[0] < 0.01 and numbers[1] > 0.99):
        return True
    return False

In [52]:
resolved_markets = []
for event in events:
    for market in event["markets"]:
        if is_resolved(market):
            resolved_markets.append(market)

print(resolved_markets)

[{'id': '2063852', 'question': 'XRP Up or Down - April 24, 8:00PM-8:05PM ET', 'conditionId': '0xd10a6e4ace4d193f41002801b52f5dec5d29578e048e2d0e34984f5c7390ceb8', 'slug': 'xrp-updown-5m-1777075200', 'resolutionSource': 'https://data.chain.link/streams/xrp-usd', 'endDate': '2026-04-25T00:05:00Z', 'liquidity': '0', 'startDate': '2026-04-24T00:08:27.112431Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/XRP-logo.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/XRP-logo.png', 'description': 'This market will resolve to "Up" if the XRP price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the XRP/USD data stream available at https://data.chain.link/streams/xrp-usd.\nPlease note that this market is about the price according to Chainlink data stream XRP/USD, not ac

In [53]:
for e in events:
    print(e["title"], "|", e.get("closedTime", "no closedTime field"))

XRP Up or Down - April 24, 8:00PM-8:05PM ET | 2026-04-25T00:05:26Z
BNB Up or Down - April 24, 8:00PM-8:05PM ET | 2026-04-25T00:05:26Z
Solana Up or Down - April 24, 8:00PM-8:05PM ET | 2026-04-25T00:05:26Z
Hyperliquid Up or Down - April 24, 8:00PM-8:05PM ET | 2026-04-25T00:05:26Z
Bitcoin Up or Down - April 24, 8:00PM-8:05PM ET | 2026-04-25T00:05:26Z


In [68]:
TOP_LEVEL_CATEGORIES = [
    "Politics",
    "Crypto",
    "Finance",
    "Sports",
    "Weather",
    "Pop Culture",
    "Science",
    "Geopolitics",
]

def classify_event(event):
    tag_labels = [t.get("label", "") for t in event.get("tags", [])]
    
    # Map noisy tag labels to clean top-level categories
    label_to_category = {
        "Politics": "Politics",
        "Crypto": "Crypto",
        "Bitcoin": "Crypto",
        "Ethereum": "Crypto",
        "Finance": "Finance",
        "Stocks": "Finance",
        "Equities": "Finance",
        "Stock Prices": "Finance",
        "Sports": "Sports",
        "NFL": "Sports",
        "NBA": "Sports",
        "Tennis": "Sports",
        "Soccer": "Sports",
        "Weather": "Weather",
        "Pop Culture": "Pop Culture",
        "Celebrities": "Pop Culture",
        "Geopolitics": "Geopolitics",
    }

    for tag in tag_labels:
        if tag in label_to_category:
            return label_to_category[tag]
    
    return "Other"

In [ ]:
def extract_market_row(market, event):
    prices = json.loads(market["outcomePrices"])
    yes_price = float(prices[0])
    no_price = float(prices[1])

    return {
        "market_id": market["id"],
        "question": market["question"],
        "category": classify_event(event),
        "yes_resolved": 1 if yes_price > 0.99 else 0,
        "created_at": market["createdAt"],
        "closed_time": event.get("closedTime", None),
        "end_date": market["endDate"],
        "volume": float(market.get("volume", 0)),
        "volume24hr": float(market.get("volume24hr", 0)),
        "volume1wk": float(market.get("volume1wk", 0)),
        "volume1mo": float(market.get("volume1mo", 0)),
        "last_trade_price": float(market.get("lastTradePrice", 0)),
        "one_day_price_change": float(market.get("oneDayPriceChange", 0)),
        "one_week_price_change": float(market.get("oneWeekPriceChange", 0)),
        "one_month_price_change": float(market.get("oneMonthPriceChange", 0))
    }

In [63]:
all_rows = []
offset = 0
PAGE_SIZE = 100
TARGET_RESOLVED = 50

while len(all_rows) < TARGET_RESOLVED:
    events = fetch_events_page(offset, PAGE_SIZE)
    if not events:
        print("No more events returned, stopping")
        break
    for event in events:
        for market in event.get("markets", []):
            if is_resolved(market):
                all_rows.append(extract_market_row(market, event))
    offset += PAGE_SIZE
    print(f"Offset {offset}: have {len(all_rows)} resolved markets")

df = pd.DataFrame(all_rows)
print()
print("Shape:", df.shape)
print()
print("Categories:")
print(df["category"].value_counts())

Offset 100: have 230 resolved markets

Shape: (230, 15)

Categories:
category
Unknown    230
Name: count, dtype: int64


In [62]:
df

,market_id,question,category,yes_resolved,created_at,closed_time,end_date,volume,volume24hr,volume1wk,volume1mo,last_trade_price,one_day_price_change,one_week_price_change,one_month_price_change
0,2008892,Will Apple (AAPL) finish week of April 20 abov...,Unknown,1,2026-04-17T22:00:01.868818Z,2026-04-25T00:26:36Z,2026-04-24T20:00:00Z,279.139500,0.000000,0.000000,0.000000,0.999,0.0700,0.0945,0.0
1,2008893,Will Apple (AAPL) finish week of April 20 abov...,Unknown,1,2026-04-17T22:00:02.326248Z,2026-04-25T00:26:36Z,2026-04-24T20:00:00Z,285.295000,0.000000,0.000000,0.000000,0.999,0.0085,0.0845,0.0
2,2008897,Will Apple (AAPL) finish week of April 20 abov...,Unknown,1,2026-04-17T22:00:02.518496Z,2026-04-25T00:26:36Z,2026-04-24T20:00:00Z,210.043000,164.993000,210.043000,210.043000,0.999,0.0665,0.0895,0.0
3,2008900,Will Apple (AAPL) finish week of April 20 abov...,Unknown,1,2026-04-17T22:00:02.924071Z,2026-04-25T00:26:36Z,2026-04-24T20:00:00Z,170.043000,164.993000,170.043000,170.043000,0.999,0.0995,0.0945,0.0
4,2008903,Will Apple (AAPL) finish week of April 20 abov...,Unknown,1,2026-04-17T22:00:03.032992Z,2026-04-25T00:26:36Z,2026-04-24T20:00:00Z,340.364866,125.395000,340.364866,340.364866,0.999,0.0995,0.1145,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,1949737,Clermont Foot 63 vs. SC Bastia: O/U 1.5,Unknown,1,2026-04-11T20:00:04.149373Z,2026-04-24T23:56:00Z,2026-04-24T18:00:00Z,25829.500405,25821.722095,25829.500405,25829.500405,0.999,0.3045,0.3445,0.0
226,1949739,Clermont Foot 63 vs. SC Bastia: O/U 2.5,Unknown,0,2026-04-11T20:00:04.261529Z,2026-04-24T23:56:00Z,2026-04-24T18:00:00Z,23232.872448,23112.898351,23232.872448,23232.872448,0.120,-0.4445,-0.4945,0.0
227,1949741,Clermont Foot 63 vs. SC Bastia: O/U 3.5,Unknown,0,2026-04-11T20:00:04.372118Z,2026-04-24T23:56:00Z,2026-04-24T18:00:00Z,2928.984125,2876.424125,2928.984125,2928.984125,0.170,-0.2195,-0.3595,0.0
228,1949743,Clermont Foot 63 vs. SC Bastia: O/U 4.5,Unknown,0,2026-04-11T20:00:04.482777Z,2026-04-24T23:56:00Z,2026-04-24T18:00:00Z,1016.301095,1004.331095,1016.301095,1016.301095,0.090,-0.0895,-0.2995,0.0


In [66]:
# Pull a few raw events fresh
response = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"closed": "true", "limit": 3, "order": "closedTime", "ascending": "false"}
)
events = response.json()

# For each event, check both event-level and market-level category
for i, event in enumerate(events[:3]):
    print(f"Event {i}: {event.get('title', '')[:50]}")
    print(f"  Tags: {event.get('tags')}")
    print()

Event 0: Lowest temperature in Paris on April 24?
  Tags: [{'id': '84', 'label': 'Weather', 'slug': 'weather', 'forceShow': False, 'publishedAt': '2023-11-02 21:17:34.81+00', 'createdAt': '2023-11-02T21:17:34.814Z', 'updatedAt': '2026-03-09T22:33:58.564774Z', 'requiresTranslation': False}, {'id': '101757', 'label': 'Recurring', 'slug': 'recurring', 'forceShow': False, 'createdAt': '2025-02-01T00:45:26.493838Z', 'updatedAt': '2026-04-17T17:25:43.117249Z', 'requiresTranslation': False}, {'id': '102169', 'label': 'Hide From New', 'slug': 'hide-from-new', 'forceShow': False, 'createdAt': '2025-05-23T18:47:07.430147Z', 'updatedAt': '2026-04-17T20:28:28.912197Z', 'isCarousel': False, 'requiresTranslation': False}, {'id': '103730', 'label': 'Paris', 'slug': 'paris', 'forceShow': False, 'createdAt': '2026-02-11T18:06:37.12671Z', 'updatedAt': '2026-04-17T17:25:12.611463Z', 'isCarousel': False, 'requiresTranslation': False}, {'id': '103040', 'label': 'Daily Temperature', 'slug': 'daily-temperatu